# Data Profiling 

### Contents

#### Correctness and Completeness 
 - Summarizes table contents as the first verification step in determining that the synthetic data generator ran and loaded correctly
 - Flagged discovered cleaning tasks 
 - Confirms no tables need to be added to the predetermined in-scope list and determines what changes need to be made 

#### Primary Key Uniqueness and Referential Integrity 
- Second verification step confirming this potential issue mentioned in the Synthea documents is not present

#### Project Viability Estimation 
- Estimates whether the population's relevant to the measure's are large enough to continue with cleaning, ingestion and measure derivation
- Inserts into codebook that stores relevant codes for medications, procedures, conditions and labs
- Payer coverage and single-payer assumption: Confirms all 65+ members have active coverage in the measurement period and that ~99% are on a government Medicare-family payer, justifying the modeling assumption that the full 65+ population is treated as enrolled in one Medicare Advantage plan

#### Summary 
- Describes codebook 
- Lists by table the cleaning tasks flagged throughout profiling for the tables that will be used going forward
- General notes about the data 


Data Dictionary: https://github.com/synthetichealth/synthea/wiki/CSV-File-Data-Dictionary

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

here = Path.cwd()
PROJECT_ROOT = here if (here / "ma_quality.duckdb").exists() else here.parent
DB = (PROJECT_ROOT / "ma_quality.duckdb").resolve()
assert DB.exists(), f"DB not found at {DB}"
con = duckdb.connect(str(DB), read_only=False)

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [2]:
in_scope = ['raw_patients', 'raw_encounters', 'raw_conditions', 'raw_observations', 'raw_procedures', 
            'raw_medications']

out_scope = ['raw_immunizations', 'raw_imaging_studies', 'raw_claims', 'raw_claims_transactions', 
            'raw_providers', 'raw_organizations', 'raw_careplans', 'raw_allergies', 'raw_devices',
            'raw_supplies', 'raw_payer_transitions', 'raw_payers']

AS_OF = "2025-12-31"

# 65+ population
con.execute('''
    CREATE OR REPLACE VIEW seniors AS
    SELECT *
    FROM raw_patients
    WHERE Birthdate <= DATE '1960-12-31'   
''')



In [3]:
# Codebook to reference relevant condition, lab and medication codes
con.execute("""
    CREATE OR REPLACE TABLE codebook (
        concept      VARCHAR,
        code         VARCHAR,
        code_system  VARCHAR,
        source_table VARCHAR,
        description  VARCHAR,
        note         VARCHAR
    )
""")


## Generation Correctness and Completeness

In [4]:
print(f"\n{'='*50}\nOUT OF SCOPE\n{'='*50}")

for t in out_scope:
    print(f"\n{'='*50}\n{t}\n{'='*50}")
    display(con.execute(f"SUMMARIZE {t}").df())

print(f"\n{'='*50}\nIN SCOPE\n{'='*50}")

for i in in_scope:
    print(f"\n{'='*50}\n{i}\n{'='*50}")
    display(con.execute(f"SUMMARIZE {i}").df())


OUT OF SCOPE

raw_immunizations


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,DATE,TIMESTAMP WITH TIME ZONE,2012-02-19 18:05:07-06,2027-02-14 17:59:20-06,478851,2019-10-07 04:57:15.818587-05,NaN,2016-03-24 06:19:11.111219-05,2020-05-19 03:26:12.933586-05,2022-12-31 05:53:56.000465-06,591376,0.0
1,PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,NaN,NaN,NaN,NaN,NaN,591376,0.0
2,ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-07e4-6267722501e6,fff5352f-2b53-d86c-fdcf-8582350378f2,473798,NaN,NaN,NaN,NaN,NaN,591376,0.0
3,CODE,BIGINT,33,212,11,139.90030200752145,29.92669815693516,140,140,140,591376,0.0
4,DESCRIPTION,VARCHAR,COVID-19 mRNA LNP-S PF 100 mcg/0.5mL dose or 50 mcg/0.25mL dose,zoster vaccine recombinant,12,NaN,NaN,NaN,NaN,NaN,591376,0.0
5,BASE_COST,DOUBLE,136.0,136.0,1,136.0,0.0,136.0,136.0,136.0,591376,0.0



raw_imaging_studies


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Id,VARCHAR,0003b2f9-73d5-3dcd-0045-bc87dae635b2,fff5352f-2b53-d86c-fe03-b6b4a7604c20,343935,NaN,NaN,NaN,NaN,NaN,704039,0.0
1,DATE,TIMESTAMP WITH TIME ZONE,1944-07-13 09:42:57-05,2027-02-14 06:35:56-06,459396,2021-05-04 19:53:37.177296-05,NaN,2018-02-19 19:22:17.062805-06,2022-12-20 20:23:31.298304-06,2025-05-15 16:01:03.45275-05,704039,0.0
2,PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,NaN,NaN,NaN,NaN,NaN,704039,0.0
3,ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-0098-a8a6ec394d02,fff5352f-2b53-d86c-86cf-63de0fe9e325,310468,NaN,NaN,NaN,NaN,NaN,704039,0.0
4,SERIES_UID,VARCHAR,1.2.840.99999999.1.10000142.1626443778278,1.2.840.99999999.1.99999722.1696479392810,367467,NaN,NaN,NaN,NaN,NaN,704039,0.0
5,BODYSITE_CODE,BIGINT,344001,732167000,15,295391445.79003435,276487384.5471216,51185008,181171005,700016008,704039,0.0
6,BODYSITE_DESCRIPTION,VARCHAR,Ankle region structure (body structure),Wrist region structure (body structure),13,NaN,NaN,NaN,NaN,NaN,704039,0.0
7,MODALITY_CODE,VARCHAR,CR,US,6,NaN,NaN,NaN,NaN,NaN,704039,0.0
8,MODALITY_DESCRIPTION,VARCHAR,Computed Radiography,Ultrasound,6,NaN,NaN,NaN,NaN,NaN,704039,0.0
9,INSTANCE_UID,VARCHAR,1.2.840.99999999.1.1.10000534.1753719618100,1.2.840.99999999.1.99.99983092.1791411416667,772995,NaN,NaN,NaN,NaN,NaN,704039,0.0



raw_claims


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,START,TIMESTAMP WITH TIME ZONE,2012-02-01 15:14:45-06,2027-02-14 17:59:20-06,7984959,2020-04-17 01:38:35.943244-05,NaN,2016-09-11 04:11:51.134132-05,2020-08-15 18:49:25.509907-05,2024-01-20 11:27:41.395352-06,11672627,0.00
1,STOP,TIMESTAMP WITH TIME ZONE,2012-02-18 18:24:02-06,2027-02-18 11:30:55-06,9438428,2020-04-17 02:15:13.279891-05,NaN,2016-09-11 05:45:20.372565-05,2020-08-15 20:31:41.556726-05,2024-01-20 11:33:52.35229-06,11672627,0.00
2,PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,NaN,NaN,NaN,NaN,NaN,11672627,0.00
3,ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-0098-a8a6ec394d02,fff5352f-2b53-d86c-fdcf-8582350378f2,2453020,NaN,NaN,NaN,NaN,NaN,11672627,0.00
4,SYSTEM,VARCHAR,SNOMED-CT,SNOMED-CT,1,NaN,NaN,NaN,NaN,NaN,11672627,0.00
5,CODE,BIGINT,3457005,16335031000119103,532,22779974807315.47,143924241218434.28,171207006,300542144,711390942,11672627,0.00
6,DESCRIPTION,VARCHAR,12 lead electrocardiogram (procedure),Weaning from mechanically assisted ventilation (regime/therapy),417,NaN,NaN,NaN,NaN,NaN,11672627,0.00
7,BASE_COST,DOUBLE,0.28,260035.95,207193,531.0246131067247,1349.253709937385,370.2,370.2,370.2,11672627,0.00
8,REASONCODE,BIGINT,1734006,368581000119106,91,541793568971.2004,8209561799398.669,66383009,98297522,103904445,11672627,60.76
9,REASONDESCRIPTION,VARCHAR,Abnormal findings diagnostic imaging heart+coronary circulat (finding),Tubal pregnancy (disorder),101,NaN,NaN,NaN,NaN,NaN,11672627,60.76



raw_medications


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,START,TIMESTAMP WITH TIME ZONE,1941-03-27 00:13:30-06,2027-02-14 17:59:20-06,1520571,2020-06-13 18:06:05.953756-05,NaN,2017-02-13 11:19:52.140787-06,2021-03-11 08:15:48.896391-06,2024-06-08 16:16:19.92736-05,4117058,0.00
1,STOP,TIMESTAMP WITH TIME ZONE,2012-02-17 03:46:17-06,2027-02-14 17:59:20-06,1421828,2020-11-18 14:02:38.655414-06,NaN,2017-07-11 20:10:03.454846-05,2021-06-02 00:50:27.500294-05,2024-07-20 07:28:22.251192-05,4117058,4.59
2,PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,NaN,NaN,NaN,NaN,NaN,4117058,0.00
3,PAYER,VARCHAR,0133f751-9229-3cfd-815f-b6d4979bdd6a,e03e23c9-4df1-3eb6-a62d-f70f02301496,10,NaN,NaN,NaN,NaN,NaN,4117058,0.00
4,ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-03ca-a9f8fbddd451,fff5352f-2b53-d86c-fdcf-8582350378f2,1936353,NaN,NaN,NaN,NaN,NaN,4117058,0.00
5,CODE,BIGINT,105585,2563431,286,510318.63006617833,464447.26409053756,205923,310933,856078,4117058,0.00
6,DESCRIPTION,VARCHAR,0.25 ML Leuprolide Acetate 30 MG/ML Prefilled Syringe,{7 (ethinyl estradiol 0.035 MG / norgestimate 0.18 MG Oral Tablet) / 7 (ethinyl estradiol 0.035 MG / norgestimate 0.215 MG Oral Tablet) / 7 (ethinyl estradiol 0.035 MG / norgestimate 0.25 MG Oral Tablet) / 7 (inert ingredients 1 MG Oral Tablet) } Pack [Trinessa 28 Day],507,NaN,NaN,NaN,NaN,NaN,4117058,0.00
7,BASE_COST,DOUBLE,0.01,211629.6,140691,196.46841851398545,2667.524935584828,1.86,59.47,264.53,4117058,0.00
8,PAYER_COVERAGE,DOUBLE,0.0,211629.6,102598,140.7278390295941,1980.63861115179,0.7784625830399193,21.87396298949228,211.62,4117058,0.00
9,DISPENSES,BIGINT,1,1045,1179,7.475354002785484,42.089857619180705,1,1,3,4117058,0.00


1. All tables generated in adherence to the documented data dictionary
2. No generation completeness issues

Tables needed: 
- Patients: patient information to determine demographic assingment and measure eligibility
- Encounters: use to derive care engagement measures
- Conditions: use to derive health related measures
- Observation: source of clinical facts for health related measures
- Medications: use to derive medication adherence measures
- Procedures: use to derive procedure related measures

### Cleaning Notes

Following code columns need to be type-casted to VARCHAR : raw_medications, procedures (both code and reasoncode), conditions, encounters reasoncode. 

Date ranges in encounters, procedures and payer_transitions run into 2027 which is expected because appointments and transitions can be scheduled ahead but mean that measures involving dates should be
bound between 2026-01-01 and 2026-12-31. 

Medications 286 codes to 507 descriptions so distinct codes match to multiple descriptions implying description case differences. 
Procedures 296 codes to 299 descriptions implying code case differences although differences may be due to using approximent unique counts.

Allergies-Stop is 100% null by Synthea design. They designed all allergies to be permanent so we can drop this column

Type-cast the following columns from timestamp to date: start_date and stop_date columns in payer_transitions, date-observations, encounters, procedures and medications start and stop columns also rename to start_date, stop_date

The following tables Id column: careplans, claims, encounters, organizations, patients, payers, providers
Claims transactions is ID instead

## Primary Key Uniqueness and Referential Integrity    

In [5]:
PRIMARY_KEYS = {
    "raw_careplans":     "Id",
    "raw_claims":        "Id",
    "raw_claims_transactions": "ID",
    "raw_encounters":    "Id",
    "raw_organizations": "Id",
    "raw_patients":      "Id",
    "raw_payers":        "Id",
    "raw_providers":     "Id",
    # NOTE: imaging_studies.Id is explicitly NON-unique per the dictionary
    # "An imaging study may have multiple rows" — excluded deliberately.
}
 
# Foreign keys: (child_table, child_col, parent_table, parent_col, spec_required)
# Grouped by parent tables
FOREIGN_KEYS = [
    # Patient FK — required on every child table
    ("raw_allergies",      "PATIENT",   "raw_patients", "Id", True),
    ("raw_careplans",      "PATIENT",   "raw_patients", "Id", True),
    ("raw_claims",         "PATIENTID", "raw_patients", "Id", True),
    ("raw_claims_transactions", "PATIENTID", "raw_patients", "Id", True),
    ("raw_conditions",     "PATIENT",   "raw_patients", "Id", True),
    ("raw_devices",        "PATIENT",   "raw_patients", "Id", True),
    ("raw_encounters",     "PATIENT",   "raw_patients", "Id", True),
    ("raw_imaging_studies","PATIENT",   "raw_patients", "Id", True),
    ("raw_immunizations",  "PATIENT",   "raw_patients", "Id", True),
    ("raw_medications",    "PATIENT",   "raw_patients", "Id", True),
    ("raw_observations",   "PATIENT",   "raw_patients", "Id", True),
    ("raw_procedures",     "PATIENT",   "raw_patients", "Id", True),
    ("raw_supplies",       "PATIENT",   "raw_patients", "Id", True),
    ("raw_payer_transitions","PATIENT", "raw_patients", "Id", True),
 
    # Encounter FK — dictionary marks ALL of these required=true
    ("raw_allergies",      "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_careplans",      "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_conditions",     "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_devices",        "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_imaging_studies","ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_immunizations",  "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_medications",    "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_observations",   "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_procedures",     "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_supplies",       "ENCOUNTER", "raw_encounters", "Id", True),
    ("raw_claims",         "APPOINTMENTID", "raw_encounters", "Id", False),
    ("raw_claims_transactions", "APPOINTMENTID", "raw_encounters", "Id", False),
 
    # Payer FK
    ("raw_encounters",       "PAYER", "raw_payers", "Id", True),
    ("raw_medications",      "PAYER", "raw_payers", "Id", True),
    ("raw_payer_transitions","PAYER", "raw_payers", "Id", True),
    ("raw_payer_transitions","SECONDARY_PAYER", "raw_payers", "Id", False),
    ("raw_claims",           "PRIMARYPATIENTINSURANCEID", "raw_payers", "Id", False),
    ("raw_claims",           "SECONDARYPATIENTINSURANCEID", "raw_payers", "Id", False),
 
    # Organization / Provider FK
    ("raw_encounters", "ORGANIZATION", "raw_organizations", "Id", True),
    ("raw_encounters", "PROVIDER",     "raw_providers",     "Id", True),
    ("raw_providers",  "ORGANIZATION", "raw_organizations", "Id", True),
    ("raw_claims",     "PROVIDERID",            "raw_providers", "Id", True),
    ("raw_claims",     "REFERRINGPROVIDERID",   "raw_providers", "Id", False),
    ("raw_claims",     "SUPERVISINGPROVIDERID", "raw_providers", "Id", False),
    ("raw_claims_transactions", "PROVIDERID",            "raw_providers",     "Id", True),
    ("raw_claims_transactions", "SUPERVISINGPROVIDERID", "raw_providers",     "Id", False),
    ("raw_claims_transactions", "PLACEOFSERVICE",        "raw_organizations", "Id", True),
 
    # Claim FK
    ("raw_claims_transactions", "CLAIMID", "raw_claims", "Id", True),
# NOTE: claims_transactions.PATIENTINSURANCEID is documented as referencing
# payer_transitions.MEMBERID, but this cannot be validated as a referential-integrity
# constraint. payer_transitions has no primary key, and MEMBERID is not unique
# its approximate distinct count is far below the row count so there is no unique
# parent key to join against. Excluded from the FK check for this reason.
]
 

In [6]:
schema = con.execute("""
    SELECT table_name, column_name
    FROM information_schema.columns
    WHERE table_name LIKE 'raw_%'
""").df()
COLS = schema.groupby("table_name")["column_name"].apply(set).to_dict()


# --- Primary key uniqueness ---
pk_sql = " UNION ALL ".join(f"""
SELECT '{table}' AS tbl, 
       '{col}' AS pk, 
       count(*) AS rows,
       count(*) - count(DISTINCT {col}) AS dupes,
       count(*) FILTER (WHERE {col} IS NULL) AS null_pk
FROM {table}
""" for table, col in PRIMARY_KEYS.items())

pk_results = con.execute(pk_sql + " ORDER BY dupes DESC, null_pk DESC").df()
display(pk_results)


# --- Foreign key referential integrity ---
fk_sql = " UNION ALL ".join(f"""
SELECT '{child}' AS child, '{child_col}' AS fk_col, '{parent}' AS parent,
       {required} AS spec_required,
       count(*) AS rows,
       count(*) FILTER (WHERE c.{child_col} IS NOT NULL AND p.{parent_col} IS NULL) AS orphans
FROM {child} c
LEFT JOIN {parent} p ON c.{child_col} = p.{parent_col}
""" for child, child_col, parent, parent_col, required in FOREIGN_KEYS)

fk_results = con.execute(fk_sql + " ORDER BY orphans DESC").df()
display(fk_results)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tbl,pk,rows,dupes,null_pk
0,raw_careplans,Id,154242,0,0
1,raw_claims,Id,7722284,0,0
2,raw_claims_transactions,ID,73457075,0,0
3,raw_encounters,Id,3605226,0,0
4,raw_organizations,Id,212,0,0
5,raw_patients,Id,30000,0,0
6,raw_payers,Id,10,0,0
7,raw_providers,Id,212,0,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,child,fk_col,parent,spec_required,rows,orphans
0,raw_careplans,PATIENT,raw_patients,True,154242,0
1,raw_encounters,ORGANIZATION,raw_organizations,True,3605226,0
2,raw_claims,PATIENTID,raw_patients,True,7722284,0
3,raw_claims_transactions,PATIENTID,raw_patients,True,73457075,0
4,raw_conditions,PATIENT,raw_patients,True,2399532,0
5,raw_devices,PATIENT,raw_patients,True,424449,0
6,raw_encounters,PATIENT,raw_patients,True,3605226,0
7,raw_imaging_studies,PATIENT,raw_patients,True,704039,0
8,raw_immunizations,PATIENT,raw_patients,True,591376,0
9,raw_medications,PATIENT,raw_patients,True,4117058,0


- Pre-set primary keys are correct having no duplicates or nulls 
- No orphaned foreign keys 
- The in-scope tables conditions, procedures, medications and observations have no primary key and will not have one created. They are fact tables that as per the data dictionary have no foreign keys referencing them and they contain foreign key columns for joins. 
- Data dictionary shows claims_transactions.patientinsuranceid is a forgeign key to payer_transitions.memberid but memberid is not unique so this relationship is not enforceable. Only something to note because neither table is used in the analysis.


## Data Sufficiency Estimation

### Age and Geographic

In [7]:
display(con.execute(f'''
    WITH age AS (
      SELECT Birthdate, Deathdate,
             date_diff('year', CAST(Birthdate AS DATE), DATE '{AS_OF}') AS age
      FROM raw_patients
    )
    SELECT count(*) AS all_patients,
           count(*) FILTER (WHERE Deathdate IS NULL) AS living,
           count(*) FILTER (WHERE Deathdate IS NULL AND age >= 65) AS living_65plus,
           count(*) FILTER (WHERE Deathdate IS NOT NULL
                              AND CAST(Deathdate AS DATE) < CAST(Birthdate AS DATE)) AS death_before_birth,
           max(Deathdate) AS max_death
    FROM age
''').df())

display(con.execute(f'''
    WITH age AS (
      SELECT 
             date_diff('year', CAST("Birthdate" AS DATE), DATE '{AS_OF}') AS age
      FROM raw_patients WHERE "Deathdate" IS NULL
    )
    SELECT CASE
             WHEN age < 65 THEN '<65'
             WHEN age BETWEEN 65 AND 69 THEN '65-69'
             WHEN age BETWEEN 70 AND 74 THEN '70-74'
             WHEN age BETWEEN 75 AND 79 THEN '75-79'
             WHEN age BETWEEN 80 AND 84 THEN '80-84'
             ELSE '85+'
           END AS age_band,
           count(*) AS total
    FROM age GROUP BY age_band ORDER BY age_band
''').df())

,all_patients,living,living_65plus,death_before_birth,max_death
0,30000,30000,28548,0,None


,age_band,total
0,65-69,7125
1,70-74,7224
2,75-79,7250
3,80-84,6949
4,<65,1452


We are using 65+ as a naive Medicare Advantage restriction. This output shows 1452 people are not 65+ so we need to create view that restricts patients to those who are 65+ to reference in the following data sufficency checks. This view will be put at the top of the notebook.

Drop the 1452 people under 65 during cleaning.

In [8]:
# Geographic grain evidence: seniors per county

display(con.execute(f'''
    SELECT p.County, count(*) AS seniors
    FROM raw_patients p
    JOIN seniors s ON s.Id = p.Id
    GROUP BY 1 ORDER BY 2 DESC
''').df())

display(con.execute('''
    SELECT p.County, count(DISTINCT p.Fips) AS n_fips
    FROM raw_patients p
    JOIN seniors s ON s.Id = p.Id
    WHERE p.County IS NOT NULL
    GROUP BY 1 
    ORDER BY 2 DESC
''').df())

display(con.execute('''
    SELECT p.County, p.Fips, count(*) as total
    FROM raw_patients p
    JOIN seniors s ON s.Id = p.Id
    WHERE p.County IS NOT NULL
    GROUP BY 1, 2 
    ORDER BY 2 DESC
''').df())

,COUNTY,seniors
0,Polk County,22070
1,Dallas County,3499
2,Warren County,1727
3,Jasper County,1252


,COUNTY,n_fips
0,Dallas County,4
1,Polk County,4
2,Jasper County,3
3,Warren County,3


,COUNTY,FIPS,total
0,Warren County,19181,1520
1,Polk County,19181,1704
2,Dallas County,19181,160
3,Polk County,19169,14
4,Jasper County,19157,3
5,Polk County,19153,19007
6,Warren County,19153,193
7,Jasper County,19153,1
8,Dallas County,19153,1383
9,Dallas County,19121,47


- Polk and Dallas should have enough patients for each measure but Warren and Jasper could thin out quickly
- Fips are too thin as is 
- Null Fips will be investigated during cleaning. Not an issue as of now since they are a small portion of the population, have a county and Fips will not be a geographic grain used.

### Measure Counts and Insurance Enrollment 

#### Regex code searching

In [9]:
# Complication-only diabetics (DKD, microalbuminuria with no base code) are
# material — union base + complication codes (D15). 
display(con.execute('''
    SELECT path,
           COALESCE(county, 'ALL COUNTIES') AS county,
           count(DISTINCT patient) AS patients
    FROM (
      SELECT 'base_or_complication' AS path, c.Patient AS patient, s.County AS county
      FROM raw_conditions c
      JOIN seniors s ON s.Id = c.Patient
      WHERE regexp_matches(lower(c.Description),
            'diabet|diabetic kidney|diabetic nephropathy|microalbumin')
        AND NOT regexp_matches(lower(c.Description), 'prediabet')
      UNION ALL
      SELECT 'pharmacy_diabetes_rx', m.Patient, s.County
      FROM raw_medications m
      JOIN seniors s ON s.Id = m.Patient
      WHERE regexp_matches(lower(m.Description),
            'metformin|glipizide|glimepiride|insulin|sitagliptin|empagliflozin')
    ) u
    GROUP BY path, GROUPING SETS ((county), ())
    ORDER BY path, (county IS NOT NULL), patients DESC
''').df())

,path,county,patients
0,base_or_complication,ALL COUNTIES,10408
1,base_or_complication,Polk County,8075
2,base_or_complication,Dallas County,1254
3,base_or_complication,Warren County,611
4,base_or_complication,Jasper County,468
5,pharmacy_diabetes_rx,ALL COUNTIES,7307
6,pharmacy_diabetes_rx,Polk County,5644
7,pharmacy_diabetes_rx,Dallas County,882
8,pharmacy_diabetes_rx,Warren County,442
9,pharmacy_diabetes_rx,Jasper County,339


Measures EED, GSD and HED should use diagnosis denominator and measues MPF, diabetes adherence and SUPD should use pharmacy denominator.

In [10]:
# --- DIABETES: check what 'diabet|glycem' catches, and the prediabetes exclusion ---
display(con.execute('''
    SELECT DISTINCT "Code", "Description", count(*) AS n
    FROM raw_conditions
    WHERE regexp_matches(lower("Description"), 'diabet|glycem|glucose|hyperglyc|hba1c')
    GROUP BY 1,2 ORDER BY n DESC
''').df())

,CODE,DESCRIPTION,n
0,714628002,Prediabetes (finding),19273
1,127013003,Disorder of kidney due to diabetes mellitus (disorder),7318
2,90781000119102,Microalbuminuria due to type 2 diabetes mellitus (disorder),6402
3,44054006,Diabetes mellitus type 2 (disorder),5776
4,157141000119108,Proteinuria due to type 2 diabetes mellitus (disorder),4665
5,80394007,Hyperglycemia (disorder),2847
6,368581000119106,Neuropathy due to type 2 diabetes mellitus (disorder),2082
7,1551000119108,Nonproliferative diabetic retinopathy due to type II diabetes mellitus,1848
8,97331000119101,Macular edema and retinopathy due to type 2 diabetes mellitus (disorder),197
9,1501000119109,Proliferative diabetic retinopathy due to type II diabetes mellitus,134


In [11]:
# HYPERTENSION: catch all to see pulmonary/other variants to exclude 
display(con.execute('''
    SELECT DISTINCT "Code", "Description", count(*) AS n
    FROM raw_conditions
    WHERE regexp_matches(lower("Description"), 'hypertens|blood pressure|htn')
    GROUP BY 1,2 ORDER BY n DESC
''').df())

,CODE,DESCRIPTION,n
0,59621000,Essential hypertension (disorder),14514


In [12]:
# CVD: catch all
display(con.execute('''
    SELECT DISTINCT "Code", "Description", count(*) AS n
    FROM raw_conditions
    WHERE regexp_matches(lower("Description"),
          'myocardial|coronary|atheroscler|cerebrovasc|ischemic|angina|infarct|stroke|heart')
    GROUP BY 1,2 ORDER BY n DESC
''').df())

,CODE,DESCRIPTION,n
0,274531002,Abnormal findings diagnostic imaging heart+coronary circulat (finding),19572
1,414545008,Ischemic heart disease (disorder),19513
2,399261000,History of coronary artery bypass grafting (situation),7803
3,399211009,History of myocardial infarction (situation),5596
4,22298006,Myocardial infarction (disorder),3873
5,401314000,Acute non-ST segment elevation myocardial infarction (disorder),3105
6,401303003,Acute ST segment elevation myocardial infarction (disorder),2220
7,88805009,Chronic congestive heart failure (disorder),528
8,230690007,Cerebrovascular accident (disorder),455
9,4557003,Preinfarction syndrome (disorder),273


In [13]:
# STATINS
display(con.execute('''
    SELECT DISTINCT "Code", "Description", count(*) AS n
    FROM raw_medications
    WHERE regexp_matches(lower("Description"), 'statin')
    GROUP BY 1,2 ORDER BY n DESC
''').df())

,CODE,DESCRIPTION,n
0,314231,Simvastatin 10 MG Oral Tablet,135038
1,312961,Simvastatin 20 MG Oral Tablet,18619
2,617311,atorvastatin 40 MG Oral Tablet,990
3,617310,atorvastatin 20 MG Oral Tablet,680
4,617312,atorvastatin 10 MG Oral Tablet,456
5,198211,simvastatin 40 MG Oral Tablet,439
6,259255,atorvastatin 80 MG Oral Tablet,329
7,312961,simvastatin 20 MG Oral Tablet,290
8,904475,pravastatin sodium 40 MG Oral Tablet,247
9,859747,rosuvastatin calcium 10 MG Oral Tablet,163


In [14]:
# RAS drugs for Hypertension Adherence 
display(con.execute('''
    SELECT DISTINCT CAST("Code" AS VARCHAR) AS code, "Description", count(*) AS n
    FROM raw_medications
    WHERE regexp_matches(lower("Description"),
          'pril|sartan|lisinopril|enalapril|ramipril|benazepril|losartan|valsartan|olmesartan|irbesartan|candesartan|telmisartan|sacubitril')
    GROUP BY 1,2 ORDER BY n DESC
''').df())


,code,DESCRIPTION,n
0,314076,lisinopril 10 MG Oral Tablet,468828
1,979485,losartan potassium 25 MG Oral Tablet,8454
2,314077,lisinopril 20 MG Oral Tablet,2116
3,979492,losartan potassium 50 MG Oral Tablet,1918
4,1656356,sacubitril 97 MG / valsartan 103 MG Oral Tablet [Entresto],483
5,979480,losartan potassium 100 MG Oral Tablet,397
6,197884,lisinopril 40 MG Oral Tablet,347
7,311354,lisinopril 5 MG Oral Tablet,332
8,311353,lisinopril 2.5 MG Oral Tablet,163
9,197885,hydrochlorothiazide 12.5 MG / lisinopril 10 MG Oral Tablet,141


In [15]:
# Diabetes drugs, covers Diabetes Adherence + SUPD pharmacy path
display(con.execute('''
    SELECT DISTINCT CAST("Code" AS VARCHAR) AS code, "Description", count(*) AS n
    FROM raw_medications
    WHERE regexp_matches(lower("Description"),
          'metformin|insulin|glipizide|glimepiride|glyburide|sitagliptin|empagliflozin|dapagliflozin|liraglutide|dulaglutide|pioglitazone')
    GROUP BY 1,2 ORDER BY n DESC
''').df())

,code,DESCRIPTION,n
0,106892,insulin isophane human 70 UNT/ML / insulin regular human 30 UNT/ML Injectable Suspension [Humulin],474801
1,860975,24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet,203890
2,865098,Insulin Lispro 100 UNT/ML Injectable Solution [Humalog],1839
3,311034,insulin regular human 100 UNT/ML Injectable Solution,999
4,897122,3 ML liraglutide 6 MG/ML Pen Injector,857


In [16]:
con.execute("""

    DELETE FROM codebook
    WHERE concept IN ('diabetes','hypertension','cvd','statin_rx','ras_rx','dm_rx');

    INSERT INTO codebook VALUES

    -- DIABETES (base + complications, D15). Excludes 714628002 prediabetes, 80394007 hyperglycemia 

    ('diabetes','44054006','SNOMED','raw_conditions','Diabetes mellitus type 2','base'),
    ('diabetes','127013003','SNOMED','raw_conditions','Disorder of kidney due to diabetes mellitus','complication (D15)'),
    ('diabetes','90781000119102','SNOMED','raw_conditions','Microalbuminuria due to type 2 diabetes mellitus','complication (D15)'),
    ('diabetes','157141000119108','SNOMED','raw_conditions','Proteinuria due to type 2 diabetes mellitus','complication (D15)'),
    ('diabetes','368581000119106','SNOMED','raw_conditions','Neuropathy due to type 2 diabetes mellitus','complication (D15)'),
    ('diabetes','1551000119108','SNOMED','raw_conditions','Nonproliferative diabetic retinopathy due to type II diabetes mellitus','complication (D15)'),
    ('diabetes','97331000119101','SNOMED','raw_conditions','Macular edema and retinopathy due to type 2 diabetes mellitus','complication (D15)'),
    ('diabetes','1501000119109','SNOMED','raw_conditions','Proliferative diabetic retinopathy due to type II diabetes mellitus','complication (D15)'),
    ('diabetes','60951000119105','SNOMED','raw_conditions','Blindness due to type 2 diabetes mellitus','complication (D15)'),

    
    -- HYPERTENSION (essential only; pulmonary excluded by absence, D17) 

    ('hypertension','59621000','SNOMED','raw_conditions','Essential hypertension','pulmonary excluded by absence (D17)'),

    
    -- CVD / ASCVD (incl. history-of situation codes, D18). Excludes heart failure, abnormal-imaging finding, postop MI n=1

    ('cvd','414545008','SNOMED','raw_conditions','Ischemic heart disease',NULL),
    ('cvd','22298006','SNOMED','raw_conditions','Myocardial infarction',NULL),
    ('cvd','401314000','SNOMED','raw_conditions','Acute non-ST elevation myocardial infarction',NULL),
    ('cvd','401303003','SNOMED','raw_conditions','Acute ST elevation myocardial infarction',NULL),
    ('cvd','230690007','SNOMED','raw_conditions','Cerebrovascular accident',NULL),
    ('cvd','4557003','SNOMED','raw_conditions','Preinfarction syndrome',NULL),
    ('cvd','399211009','SNOMED','raw_conditions','History of myocardial infarction','situation; still ASCVD (D18)'),
    ('cvd','399261000','SNOMED','raw_conditions','History of coronary artery bypass grafting','situation; still ASCVD (D18)'),

    
    -- STATINS (any statin fill; all doses/brands/combos). Dedup on code, not description 
    
    ('statin_rx','314231','RxNorm','raw_medications','Simvastatin 10 MG Oral Tablet',NULL),
    ('statin_rx','312961','RxNorm','raw_medications','Simvastatin 20 MG Oral Tablet',NULL),
    ('statin_rx','198211','RxNorm','raw_medications','Simvastatin 40 MG Oral Tablet',NULL),
    ('statin_rx','200345','RxNorm','raw_medications','Simvastatin 80 MG Oral Tablet',NULL),
    ('statin_rx','312962','RxNorm','raw_medications','Simvastatin 5 MG Oral Tablet',NULL),
    ('statin_rx','617311','RxNorm','raw_medications','Atorvastatin 40 MG Oral Tablet',NULL),
    ('statin_rx','617310','RxNorm','raw_medications','Atorvastatin 20 MG Oral Tablet',NULL),
    ('statin_rx','617312','RxNorm','raw_medications','Atorvastatin 10 MG Oral Tablet',NULL),
    ('statin_rx','259255','RxNorm','raw_medications','Atorvastatin 80 MG Oral Tablet',NULL),
    ('statin_rx','617320','RxNorm','raw_medications','Atorvastatin 40 MG Oral Tablet [Lipitor]',NULL),
    ('statin_rx','617314','RxNorm','raw_medications','Atorvastatin 10 MG Oral Tablet [Lipitor]',NULL),
    ('statin_rx','904475','RxNorm','raw_medications','Pravastatin sodium 40 MG Oral Tablet',NULL),
    ('statin_rx','904467','RxNorm','raw_medications','Pravastatin sodium 20 MG Oral Tablet',NULL),
    ('statin_rx','904481','RxNorm','raw_medications','Pravastatin sodium 80 MG Oral Tablet',NULL),
    ('statin_rx','904458','RxNorm','raw_medications','Pravastatin sodium 10 MG Oral Tablet',NULL),
    ('statin_rx','904477','RxNorm','raw_medications','Pravastatin sodium 40 MG Oral Tablet [Pravachol]',NULL),
    ('statin_rx','859747','RxNorm','raw_medications','Rosuvastatin calcium 10 MG Oral Tablet',NULL),
    ('statin_rx','859751','RxNorm','raw_medications','Rosuvastatin calcium 20 MG Oral Tablet',NULL),
    ('statin_rx','859419','RxNorm','raw_medications','Rosuvastatin calcium 40 MG Oral Tablet',NULL),
    ('statin_rx','859424','RxNorm','raw_medications','Rosuvastatin calcium 5 MG Oral Tablet',NULL),
    ('statin_rx','859753','RxNorm','raw_medications','Rosuvastatin calcium 20 MG Oral Tablet [Crestor]',NULL),
    ('statin_rx','859749','RxNorm','raw_medications','Rosuvastatin calcium 10 MG Oral Tablet [Crestor]',NULL),
    ('statin_rx','197905','RxNorm','raw_medications','Lovastatin 40 MG Oral Tablet',NULL),
    ('statin_rx','197904','RxNorm','raw_medications','Lovastatin 20 MG Oral Tablet',NULL),
    ('statin_rx','197903','RxNorm','raw_medications','Lovastatin 10 MG Oral Tablet',NULL),
    ('statin_rx','861654','RxNorm','raw_medications','Pitavastatin calcium 4 MG Oral Tablet [Livalo]',NULL),
    ('statin_rx','861650','RxNorm','raw_medications','Pitavastatin calcium 2 MG Oral Tablet [Livalo]',NULL),
    ('statin_rx','476351','RxNorm','raw_medications','Ezetimibe 10 MG / Simvastatin 80 MG Oral Tablet','combo; contains statin'),
    ('statin_rx','476349','RxNorm','raw_medications','Ezetimibe 10 MG / Simvastatin 20 MG Oral Tablet','combo; contains statin'),

        
    -- RAS drugs  for Hypertension Adherence

    ('ras_rx','314076','RxNorm','raw_medications','lisinopril 10 MG Oral Tablet','ACE'),
    ('ras_rx','314077','RxNorm','raw_medications','lisinopril 20 MG Oral Tablet','ACE'),
    ('ras_rx','197884','RxNorm','raw_medications','lisinopril 40 MG Oral Tablet','ACE'),
    ('ras_rx','311354','RxNorm','raw_medications','lisinopril 5 MG Oral Tablet','ACE'),
    ('ras_rx','311353','RxNorm','raw_medications','lisinopril 2.5 MG Oral Tablet','ACE'),
    ('ras_rx','205326','RxNorm','raw_medications','lisinopril 30 MG Oral Tablet','ACE'),
    ('ras_rx','858810','RxNorm','raw_medications','enalapril maleate 20 MG Oral Tablet','ACE'),
    ('ras_rx','858817','RxNorm','raw_medications','enalapril maleate 10 MG Oral Tablet','ACE'),
    ('ras_rx','858804','RxNorm','raw_medications','enalapril maleate 2.5 MG Oral Tablet','ACE'),
    ('ras_rx','858813','RxNorm','raw_medications','enalapril maleate 5 MG Oral Tablet','ACE'),
    ('ras_rx','261962','RxNorm','raw_medications','ramipril 10 MG Oral Capsule','ACE'),
    ('ras_rx','198188','RxNorm','raw_medications','ramipril 2.5 MG Oral Capsule','ACE'),
    ('ras_rx','198189','RxNorm','raw_medications','ramipril 5 MG Oral Capsule','ACE'),
    ('ras_rx','898719','RxNorm','raw_medications','benazepril hydrochloride 40 MG Oral Tablet','ACE'),
    ('ras_rx','898690','RxNorm','raw_medications','benazepril hydrochloride 20 MG Oral Tablet','ACE'),
    ('ras_rx','898687','RxNorm','raw_medications','benazepril hydrochloride 10 MG Oral Tablet','ACE'),
    ('ras_rx','898723','RxNorm','raw_medications','benazepril hydrochloride 5 MG Oral Tablet','ACE'),
    ('ras_rx','312748','RxNorm','raw_medications','quinapril 10 MG Oral Tablet','ACE'),
    ('ras_rx','979485','RxNorm','raw_medications','losartan potassium 25 MG Oral Tablet','ARB'),
    ('ras_rx','979492','RxNorm','raw_medications','losartan potassium 50 MG Oral Tablet','ARB'),
    ('ras_rx','979480','RxNorm','raw_medications','losartan potassium 100 MG Oral Tablet','ARB'),
    ('ras_rx','349199','RxNorm','raw_medications','valsartan 80 MG Oral Tablet','ARB'),
    ('ras_rx','349201','RxNorm','raw_medications','valsartan 160 MG Oral Tablet','ARB'),
    ('ras_rx','349200','RxNorm','raw_medications','valsartan 320 MG Oral Tablet','ARB'),
    ('ras_rx','349483','RxNorm','raw_medications','valsartan 40 MG Oral Tablet','ARB'),
    ('ras_rx','352001','RxNorm','raw_medications','valsartan 320 MG Oral Tablet [Diovan]','ARB'),
    ('ras_rx','200096','RxNorm','raw_medications','irbesartan 300 MG Oral Tablet','ARB'),
    ('ras_rx','200095','RxNorm','raw_medications','irbesartan 150 MG Oral Tablet','ARB'),
    ('ras_rx','205305','RxNorm','raw_medications','telmisartan 80 MG Oral Tablet','ARB'),
    ('ras_rx','282755','RxNorm','raw_medications','telmisartan 20 MG Oral Tablet','ARB'),
    ('ras_rx','213432','RxNorm','raw_medications','telmisartan 80 MG Oral Tablet [Micardis]','ARB'),
    ('ras_rx','1656356','RxNorm','raw_medications','sacubitril 97 MG / valsartan 103 MG Oral Tablet [Entresto]','ARNI'),
    ('ras_rx','1656346','RxNorm','raw_medications','sacubitril 24 MG / valsartan 26 MG Oral Tablet [Entresto]','ARNI'),
    ('ras_rx','197885','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / lisinopril 10 MG Oral Tablet','combo; contains ACE'),
    ('ras_rx','197887','RxNorm','raw_medications','hydrochlorothiazide 25 MG / lisinopril 20 MG Oral Tablet','combo; contains ACE'),
    ('ras_rx','197886','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / lisinopril 20 MG Oral Tablet','combo; contains ACE'),
    ('ras_rx','979471','RxNorm','raw_medications','hydrochlorothiazide 25 MG / losartan potassium 100 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','979468','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / losartan potassium 50 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','979464','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / losartan potassium 100 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','349353','RxNorm','raw_medications','hydrochlorothiazide 25 MG / valsartan 160 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','636045','RxNorm','raw_medications','hydrochlorothiazide 25 MG / valsartan 320 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','200285','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / valsartan 160 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','200284','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / valsartan 80 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','636042','RxNorm','raw_medications','hydrochlorothiazide 12.5 MG / valsartan 320 MG Oral Tablet','combo; contains ARB'),
    ('ras_rx','898342','RxNorm','raw_medications','amlodipine 10 MG / benazepril hydrochloride 20 MG Oral Capsule','combo; contains ACE'),
    ('ras_rx','898353','RxNorm','raw_medications','amlodipine 5 MG / benazepril hydrochloride 10 MG Oral Capsule','combo; contains ACE'),
    ('ras_rx','898346','RxNorm','raw_medications','amlodipine 10 MG / benazepril hydrochloride 40 MG Oral Capsule','combo; contains ACE'),
    ('ras_rx','898356','RxNorm','raw_medications','amlodipine 5 MG / benazepril hydrochloride 20 MG Oral Capsule','combo; contains ACE'),
    ('ras_rx','898350','RxNorm','raw_medications','amlodipine 2.5 MG / benazepril hydrochloride 10 MG Oral Capsule','combo; contains ACE'),
    ('ras_rx','898372','RxNorm','raw_medications','benazepril hydrochloride 20 MG / hydrochlorothiazide 25 MG Oral Tablet','combo; contains ACE'),
    ('ras_rx','898367','RxNorm','raw_medications','benazepril hydrochloride 20 MG / hydrochlorothiazide 12.5 MG Oral Tablet','combo; contains ACE'),
    ('ras_rx','898362','RxNorm','raw_medications','benazepril hydrochloride 10 MG / hydrochlorothiazide 12.5 MG Oral Tablet','combo; contains ACE'),
    ('ras_rx','999970','RxNorm','raw_medications','amlodipine / hydrochlorothiazide / olmesartan Oral Tablet [Tribenzor]','combo; contains ARB'),
    ('ras_rx','999998','RxNorm','raw_medications','amlodipine 5 / HCTZ 12.5 / olmesartan medoxomil 40 MG [Tribenzor]','combo; contains ARB'),

    
    -- DIABETES drugs for Diabetes Adherence + SUPD pharmacy path
    
    ('dm_rx','860975','RxNorm','raw_medications','24 HR Metformin hydrochloride 500 MG ER Oral Tablet','biguanide'),
    ('dm_rx','106892','RxNorm','raw_medications','insulin isophane / regular human 70/30 Injectable [Humulin]','insulin'),
    ('dm_rx','865098','RxNorm','raw_medications','Insulin Lispro 100 UNT/ML Injectable [Humalog]','insulin'),
    ('dm_rx','311034','RxNorm','raw_medications','insulin regular human 100 UNT/ML Injectable','insulin'),
    ('dm_rx','897122','RxNorm','raw_medications','liraglutide 6 MG/ML Pen Injector','GLP-1')
""")

display(con.execute("SELECT concept, count(*) AS n_codes FROM codebook GROUP BY 1 ORDER BY 1").df())

,concept,n_codes
0,cvd,8
1,diabetes,9
2,dm_rx,5
3,hypertension,1
4,ras_rx,54
5,statin_rx,29


#### Measure denominator and insurance span counts

In [17]:
display(con.execute('''
    WITH span_flag AS (
      SELECT s.Id, s.County,
             NOT EXISTS (SELECT 1 FROM raw_payer_transitions p WHERE p.Patient = s.Id) AS no_span
      FROM seniors s
    ),
    dx AS (

      -- condition-based concepts (diabetes, hypertension, cvd)
      SELECT DISTINCT c.Patient AS pid, cb.concept AS measure_grp
      FROM raw_conditions c
      JOIN codebook cb
        ON cb.source_table = 'raw_conditions'
       AND CAST(c.Code AS VARCHAR) = cb.code

      UNION

      -- medication-based concepts (statin_rx, ras_rx and dm_rx)
      SELECT DISTINCT m.Patient, cb.concept
      FROM raw_medications m
      JOIN codebook cb
        ON cb.source_table = 'raw_medications'
       AND CAST(m.Code AS VARCHAR) = cb.code
    )
    SELECT
      COALESCE(dx.measure_grp, 'NO_RELEVANT_CONDITIONS')        AS measure_grp,
      COALESCE(sf.County, 'ALL COUNTIES')               AS county,
      count(DISTINCT sf.Id)                             AS members,
      count(DISTINCT sf.Id) FILTER (WHERE sf.no_span)   AS no_span_members,
      round(100.0 * count(DISTINCT sf.Id) FILTER (WHERE sf.no_span)
            / count(DISTINCT sf.Id), 2)                 AS pct_no_span
    FROM span_flag sf
    LEFT JOIN dx ON dx.pid = sf.Id
    GROUP BY GROUPING SETS ((measure_grp, sf.County), (measure_grp))
    ORDER BY measure_grp, (county = 'ALL COUNTIES') DESC, members DESC
''').df())



,measure_grp,county,members,no_span_members,pct_no_span
0,NO_RELEVANT_CONDITIONS,Polk County,2406,0,0.0
1,NO_RELEVANT_CONDITIONS,Dallas County,375,0,0.0
2,NO_RELEVANT_CONDITIONS,Warren County,196,0,0.0
3,NO_RELEVANT_CONDITIONS,Jasper County,109,0,0.0
4,NO_RELEVANT_CONDITIONS,ALL COUNTIES,3086,0,0.0
5,cvd,Polk County,15658,0,0.0
6,cvd,Dallas County,2533,0,0.0
7,cvd,Warren County,1237,0,0.0
8,cvd,Jasper County,933,0,0.0
9,cvd,ALL COUNTIES,20361,0,0.0


In [18]:
# PCR: discover how inpatient encounters are labeled before counting 
display(con.execute('''
    SELECT DISTINCT "EncounterClass" AS enc_class, count(*) AS n
    FROM raw_encounters
    GROUP BY 1 ORDER BY n DESC
''').df())

,enc_class,n
0,ambulatory,2032070
1,wellness,630033
2,outpatient,372793
3,urgentcare,272083
4,emergency,125857
5,home,90071
6,inpatient,51332
7,snf,15173
8,hospice,10580
9,virtual,5234


In [19]:
display(con.execute('''
    WITH inpatient AS (
      SELECT DISTINCT e.Patient AS pid
      FROM raw_encounters e
      WHERE lower(e."EncounterClass") = 'inpatient'
    )
    SELECT COALESCE(s.County, 'ALL COUNTIES') AS county,
           count(DISTINCT s.Id) AS members
    FROM seniors s
    JOIN inpatient i ON i.pid = s.Id
    GROUP BY GROUPING SETS ((s.County), ())
    ORDER BY (county = 'ALL COUNTIES') DESC, members DESC
''').df())

,county,members
0,Polk County,15095
1,Dallas County,2402
2,Warren County,1153
3,Jasper County,851
4,ALL COUNTIES,19501


In [20]:
display(con.execute('''
    WITH active_span AS (
      SELECT s.Id,
             pay."NAME"      AS payer_name,
             pay."OWNERSHIP" AS ownership,
             row_number() OVER (PARTITION BY s.Id
                                ORDER BY pt."START_DATE" DESC) AS rn
      FROM seniors s
      JOIN raw_payer_transitions pt ON pt."PATIENT" = s.Id
      JOIN raw_payers pay ON pay."Id" = pt."PAYER"
      WHERE CAST(pt."START_DATE" AS DATE) <= DATE '2026-12-31'
        AND (pt."END_DATE" IS NULL OR CAST(pt."END_DATE" AS DATE) >= DATE '2026-01-01')
    )
    SELECT payer_name, ownership, count(*) AS members
    FROM active_span
    WHERE rn = 1                
    GROUP BY 1,2 ORDER BY members DESC
''').df())

,payer_name,ownership,members
0,Medicare,GOVERNMENT,24946
1,Dual Eligible,GOVERNMENT,3258
2,Medicaid,GOVERNMENT,305
3,NO_INSURANCE,NO_INSURANCE,39


Measure enrollments are large enough overall and when segmented by county (low is dm_rx Jasper county at 339)

All 65+ members have coverage in the measurement period, and 98.9% are covered by a government Medicare-family payer (87.4% Medicare, 11.4% Dual Eligible, 1.1% Medicaid), which supports the naive assumption that every member is enrolled in our Medicare Advantage plan. Synthea cannot generate a Medicare-only population, and restricting to Medicare-only members thins the measure denominators too much to support county-level analysis; with no capacity to generate additional patients, treating the full 65+ population as MA-enrolled is the next best option. 

Dual eligible patients are known to have lower closure rates and will likely bias rates to be lower than a typical Medicare Advantage population.

### Labs and Vitals

In [21]:
display(con.execute('''
    SELECT
            -- Expected codes are the standard LOINC codes
      CASE
        WHEN lower(Description) LIKE '%a1c%' OR lower(Description) LIKE '%hemoglobin a1c%' THEN 'A1c (expect 4548-4)'
        WHEN lower(Description) LIKE '%systolic%'  THEN 'Systolic (expect 8480-6)'
        WHEN lower(Description) LIKE '%diastolic%' THEN 'Diastolic (expect 8462-4)'
        WHEN lower(Description) LIKE '%glomerular%' OR lower(Description) LIKE '%egfr%' THEN 'eGFR (expect 33914-3)'
        WHEN lower(Description) LIKE '%microalbumin%' OR lower(Description) LIKE '%albumin/creatinine%'
             OR lower(Description) LIKE '%albumin%creatinine%' THEN 'uACR (expect 14959-1)'
      END AS concept,
      CAST(Code AS VARCHAR) AS code,
      Description,
      count(*) AS rows,
      count(DISTINCT Patient) AS patients
    FROM raw_observations
    WHERE lower(Description) LIKE '%a1c%'
       OR lower(Description) LIKE '%systolic%'
       OR lower(Description) LIKE '%diastolic%'
       OR lower(Description) LIKE '%glomerular%' OR lower(Description) LIKE '%egfr%'
       OR lower(Description) LIKE '%microalbumin%' OR lower(Description) LIKE '%albumin%creatinine%'
    GROUP BY concept, code, Description
    ORDER BY concept, rows DESC
''').df())

,concept,code,DESCRIPTION,rows,patients
0,A1c (expect 4548-4),4548-4,Hemoglobin A1c/Hemoglobin.total in Blood,802105,23082
1,Diastolic (expect 8462-4),8462-4,Diastolic Blood Pressure,893154,30000
2,Systolic (expect 8480-6),8480-6,Systolic Blood Pressure,893154,30000
3,eGFR (expect 33914-3),33914-3,Glomerular filtration rate [Volume Rate/Area] in Serum or Plasma by Creatinine-based formula (MDRD)/1.73 sq M,1135204,21184
4,uACR (expect 14959-1),14959-1,Microalbumin/Creatinine [Mass Ratio] in Urine,263557,5776


In [22]:
con.execute("""

    DELETE FROM codebook
    WHERE concept IN ('ked_egfr','ked_uacr','cbp_systolic','cbp_diastolic','gsd_a1c');

    INSERT INTO codebook VALUES
    ('ked_egfr','33914-3','LOINC','raw_observations','Estimated glomerular filtration rate (eGFR)','KED numerator; paired with uACR'),
    ('ked_uacr','14959-1','LOINC','raw_observations','Microalbumin/creatinine ratio (uACR)','KED numerator; back-half loaded'),
    ('cbp_systolic','8480-6','LOINC','raw_observations','Systolic blood pressure','CBP numerator; <140'),
    ('cbp_diastolic','8462-4','LOINC','raw_observations','Diastolic blood pressure','CBP numerator; <90'),
    ('gsd_a1c','4548-4','LOINC','raw_observations','Hemoglobin A1c','GSD numerator; needs numeric value')
""")

In [23]:
display(con.execute(f'''
    SELECT Code AS code,
           count(*) AS rows_all_years,
           count(*) FILTER (WHERE CAST(Date AS DATE)
                            BETWEEN DATE '2026-01-01' AND DATE '2026-06-30') AS rows_my_ytd,
           round(100.0*count(*) FILTER (WHERE "Encounter" IS NULL)/count(*),2)
             AS pct_null_encounter
    FROM raw_observations
    WHERE "Code" IN (
            SELECT code FROM codebook WHERE source_table = 'raw_observations'
    )
    GROUP BY 1 ORDER BY 1
''').df())

,code,rows_all_years,rows_my_ytd,pct_null_encounter
0,14959-1,263557,12105,0.0
1,33914-3,1135204,52136,0.0
2,4548-4,802105,35259,0.0
3,8462-4,893154,36718,0.0
4,8480-6,893154,36718,0.0


### Procedures

In [24]:
display(con.execute('''
    SELECT CAST(p.Code AS VARCHAR) AS code, 
           count(*) AS rows, 
           s.County,
           count(DISTINCT p.Patient) AS patients,
           min(CAST(CAST(p.Start AS TIMESTAMP) AS DATE)) AS earliest,
           max(CAST(CAST(p.Start AS TIMESTAMP) AS DATE)) AS latest
    FROM raw_procedures p
    JOIN seniors s ON s.Id = p.Patient
    WHERE CAST(p.Code AS VARCHAR) IN ('73761001','722161008') -- SNOMED Codes
    GROUP BY GROUPING SETS (
            (s.County, p.Code), 
            (p.Code)
            )
''').df())

,code,rows,COUNTY,patients,earliest,latest
0,73761001,57489,NaN,19052,2012-02-18,2027-02-14
1,73761001,2488,Jasper County,825,2012-02-25,2027-02-08
2,722161008,75935,NaN,4296,2012-02-19,2027-02-13
3,73761001,3507,Warren County,1165,2012-02-19,2027-02-10
4,722161008,3517,Jasper County,203,2012-02-19,2027-02-08
5,722161008,4428,Warren County,249,2012-02-23,2027-02-11
6,73761001,44557,Polk County,14755,2012-02-18,2027-02-14
7,73761001,6937,Dallas County,2307,2012-02-23,2027-02-10
8,722161008,9154,Dallas County,543,2012-02-21,2027-02-13
9,722161008,58836,Polk County,3301,2012-02-19,2027-02-13


## Close Connection

In [ ]:
con.close()

# Summary

### Codebook - source of relevant information for procedure, condition, lab and medication codes

Columns 
- concept: what grouping the code is related to (diabetes, ked_egfr, hypertension, ect)
- code: clinical code
- code_system: clinical terminology group the code belongs to (LONIC, SNOMED, ect)
- source_table: what raw table the code is drawn from 
- description: what the code measures or stands for
- note: furthur description used to differentiate similar codes or state its use


### Cleaning steps by table 


Patients:
- 174 null Fips in Polk county

Encounters:
- Type-cast reasoncode to varchar 
- Type-cast start and stop from timestamp to date and rename to _date

Conditions: 
- Type-cast code to varchar

Observations:  
- Type-cast date from timestamp to date

Procedures: 
- Type-cast both code and reasoncode to varchar
- Procedures 296 codes to 299 descriptions implying code case differences although differences may be due to using approximate unique counts
- Type-cast start and stop from timestamp to date and rename to _date

Medications:
- Type-cast code to varchar
- 286 codes to 507 descriptions so distinct codes match to multiple descriptions implying description case differences
- Type-cast start and stop from timestamp to date and rename to _date


### Notes by table

Patients:
- Smallest geographic grain is county FIPS are too thin

Medications:
- Match only on diabet adjacent regex glucose and glycem to not return true positives diabet did not already catch
- Measures EED, GSD and HED should use diagnosis denominator and measues MPF, diabetes adherence and SUPD should use pharmacy denominator.
- CVD denominators should filter to only SNOMED (disorder)

### General Notes and Cleaning

- Create a final population table to replace the seniors view
- Date ranges can run into 2027 so all filtering using dates should be bound within 2026
- Pre-set primary and foreign keys are correct and consistent with data dictionary
- Drop the 1452 patients under 65